<a href="https://colab.research.google.com/github/sarangcsoman/dataviz-exercises-sarang/blob/main/week07/lecture07_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/content/global_energy_mix.csv')

source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}

df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
# Task 1
# YOUR CODE HERE

# Filter fossil fuel sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe colours for fossil sources
fossil_colors = {
    'Coal': '#2E75B6',
    'Oil': '#E07B39',
    'Natural Gas': '#AAAAAA'
}

# Find the region with the highest fossil fuel consumption
region_totals = df_fossil.groupby('Region')['TWh'].sum()
top_region = region_totals.idxmax()

# Create treemap
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title=f'{top_region} has the highest fossil fuel consumption',
    template='plotly_white'
)

# Show TWh values, not percentages
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    marker=dict(line=dict(color='white', width=1))
)
# Grey out parent nodes (Region and Country)
fig.data[0].marker.colors = [
    fossil_colors[label] if label in fossil_colors else '#AAAAAA'
    for label in fig.data[0].labels
]

fig.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — charts
# YOUR CODE HERE

# Filter low-carbon energy sources only
df_low_carbon = df.loc[df['Source_Type'] == 'Low-carbon'].copy()

# Calculate total low-carbon energy (TWh) by country
low_carbon_country = (
    df_low_carbon.groupby('Country', as_index=False)['TWh'].sum()
)

# Add a dummy root for the treemap
low_carbon_country['All'] = 'Low-carbon'

# Find the country with the highest low-carbon energy
top_country = low_carbon_country.loc[
    low_carbon_country['TWh'].idxmax(), 'Country'
]

# Treemap
fig = px.treemap(
    low_carbon_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon energy by country',
    template='plotly_white'
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig.show()

# Horizontal bar chart
low_carbon_country = low_carbon_country.sort_values(
    'TWh', ascending=True
)

fig = px.bar(
    low_carbon_country,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#2E75B6'],
    title=f'{top_country} produces the most low-carbon energy',
    template='plotly_white'
)

fig.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside'
)

fig.show()


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*The horizontal bar chart tells the ranking story more clearly because countries can be compared easily using the lengths of their bars. The treemap is better for showing the relative size and overall distribution of low-carbon energy across countries.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3
# YOUR CODE HERE

# Aggregate fossil fuel TWh by region and source
region_mix = (
    df_fossil.groupby(['Region', 'Source'])['TWh']
    .sum()
    .reset_index()
)

# Calculate each source's share of regional fossil energy
region_mix['Share'] = (
    region_mix['TWh'] /
    region_mix.groupby('Region')['TWh'].transform('sum')
)

# Find the region with the highest coal share
coal_data = region_mix.loc[region_mix['Source'] == 'Coal']
top_coal_region = coal_data.loc[coal_data['Share'].idxmax(), 'Region']
top_coal_share = coal_data['Share'].max()

# Add percentage labels only to coal
region_mix['Label'] = np.where(
    region_mix['Source'] == 'Coal',
    region_mix['Share'].map('{:.0%}'.format),
    ''
)

# CVD-friendly colours
region_colors = {
    'Coal': fossil_colors['Coal'],
    'Oil': '#AAAAAA',
    'Natural Gas': '#DDDDDD'
}

# Sort regions by coal share, highest first
region_order = (
    coal_data.sort_values('Share', ascending=False)['Region'].tolist()
)

# Create horizontal 100% stacked bar chart
fig = px.bar(
    region_mix,
    x='Share',
    y='Region',
    color='Source',
    text='Label',
    orientation='h',
    color_discrete_map=region_colors,
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    title=f'{top_coal_region} relies most on coal ({top_coal_share:.0%} of fossil energy)',
    template='plotly_white'
)

fig.update_traces(textposition='inside')

fig.update_layout(
    barmode='stack',
    xaxis=dict(tickformat='.0%'),
    xaxis_title='Share of regional fossil energy',
    yaxis_title=''
)

# Put the highest coal-share region at the top
fig.update_yaxes(
    categoryorder='array',
    categoryarray=region_order,
    autorange='reversed'
)

fig.show()


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*The treemap shows which regions and countries consume the most fossil fuel and how consumption is divided between Coal, Oil and Natural Gas. The 100% stacked bar chart shows which region relies most on coal as a proportion of its fossil fuel consumption. Africa has the highest coal share at 78%.